In [ ]:
# code to match the responses in non-properly formated batch tweets 

# preprocessing
df_long['score'] = df_long['score'].astype(str)

# regex to match digits possibly with trailing apostrophes
pattern = r"^\s*(\d+(?:\.\d+)?)(?:')?\s*$"

# Extract the number using the regex
df_long['score'] = df_long['score'].str.extract(pattern, expand=False)
df_long['score'] = pd.to_numeric(df_long['score'])
df_long.groupby('condition')['score'].apply(lambda x: x.isna().sum())

In [ ]:
# CODE TO PRODUCE THE PREVALENCES PLOT 

# Optionally, extract the base condition (everything before any underscore).
df = df_long[df_long['shot'] == 'zero-shot'].copy()
df['base_condition'] = df['condition'].apply(lambda x: x.split('_')[0])

# use the overall HS/OL from Kern et al. (2023) as comparison
overall_OL = final_summary.loc['overall']['OL']
overall_HS = final_summary.loc['overall']['HS']


# A custom function to extract the correct group type.
def extract_group_type(x):
    if '_' not in x:
        return 'base'
    elif x.endswith('_batch_conf'):
        return 'batch_conf'
    elif x.endswith('_conf'):
        return 'conf'
    elif x.endswith('_batch'):
        return 'batch'
    else:
        return 'other'

# Create a column for the group type.
df['group_type'] = df['condition'].apply(extract_group_type)

# Melt the DataFrame to have one row per condition-measure combination.
df_melt = df.melt(
    id_vars=['condition', 'base_condition', 'group_type'],
    value_vars=['OL', 'HS'],
    var_name='Measure',
    value_name='Value'
)

# Define the desired order for our groups.
group_order = ['base', 'conf', 'batch', 'batch_conf']

# Create subplots – one subplot per group type.
fig, axes = plt.subplots(nrows=1, ncols=len(group_order), figsize=(20, 6), sharey=True)


for ax, grp in zip(axes, group_order):
    # Subset data for the current group type.
    df_grp = df_melt[df_melt['group_type'] == grp]
    
    # Create the order for conditions for the current group.
    if grp == 'base':
        cond_order = [cond for cond in custom_order if '_' not in cond]
    elif grp == 'conf':
        # Select conditions that end with '_conf' but NOT with '_batch_conf'
        cond_order = [cond for cond in custom_order if cond.endswith('_conf') and not cond.endswith('_batch_conf')]
    elif grp == 'batch':
        # Select conditions that end with '_batch' (and not batch_conf because that is handled separately)
        cond_order = [cond for cond in custom_order if cond.endswith('_batch') and not cond.endswith('_batch_conf')]
    elif grp == 'batch_conf':
        cond_order = [cond for cond in custom_order if cond.endswith('_batch_conf')]
    
    # Plot the grouped barplot for this subgroup.
    sns.barplot(
        data=df_grp,
        x='condition',
        y='Value',
        hue='Measure',
        order=cond_order,  # custom order
        dodge=True,
        ax=ax, 
        palette = {'OL': '#1f77b4', # hex colors as of jacobs paper
                   'HS': '#ff7f0e'}
    )
    
    # Add a dashed horizontal line for overall OL and HS.
    # Using the same colors as defined in the palette.
    ax.axhline(y=overall_OL, linestyle="--", color="#1f77b4", label="% OL (Kern et al.)")
    ax.axhline(y=overall_HS, linestyle="--", color="#ff7f0e", label="% HS (Kern et al.)")
    
    ax.set_title(grp)
    ax.set_xlabel("Condition")
    ax.tick_params(axis='x', rotation=90)
    ax.legend().remove()

# Set a common y-axis label.
axes[0].set_ylabel("Proportion")

# Add a single common legend.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Measure", loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Plot code for single rater agreement heatmap (single meaning outcome and few-shot/single-shot being fixed)

# Prepare combined matrix for plotting

outcome = 'OL'
shot = 'zero'


x_shot = f'{shot}-shot'
outcomes = ['OL', 'HS']

other_outcome = outcomes.copy()
other_outcome.remove(outcome)
no_cond = f'C.{other_outcome[0]}'

kappa_df = kappa_dfs[outcome][x_shot]
global_kappa_series = kappa_df.squeeze()

target_df = eval(f'across_{outcome}_{shot}')
combined_matrix = target_df.copy()

combined_matrix['Global'] = global_kappa_series

# Insert a gap column (filled with NaN values) for visual separation.
gap_col_name = ""
# Use the original condition columns order from target df:
original_conditions = list(target_df.columns)
# Insert the gap between the original condition columns and the new "Global" column.
combined_matrix[gap_col_name] = np.nan
# New order: original conditions, gap, then Global.
combined_matrix = combined_matrix[ original_conditions + [gap_col_name, 'Global'] ]

# Exclude the mismatch agreement pairs from combined matrix (e.g., Agreement between C.OL and C.HS)
combined_matrix = combined_matrix.loc[:, ~combined_matrix.columns.str.contains(no_cond)]
combined_matrix = combined_matrix[~combined_matrix.index.str.contains(no_cond)]

cols_list = list(combined_matrix)
cols_list = [x for x in cols_list if x not in ['Global', '']]
n = len(cols_list)

mask_sub = np.triu(np.ones((n, n), dtype=bool), k=1)

# For the extra two columns (gap and Global), we want no masking.
mask_extra = np.zeros((n, 2), dtype=bool)

# Concatenate the masks horizontally to match the shape of combined_matrix.
mask = np.hstack([mask_sub, mask_extra])

# Create the heatmap using the custom mask.
cmap = sns.diverging_palette(240, 10, as_cmap=True)
cmap.set_bad(color='white')  # NaN values in the gap will appear white

plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    combined_matrix,
    mask=mask,
    annot=True,     # Display numeric values
    fmt=".2f",      # Format values to 2 decimal places
    cmap='coolwarm',
    cbar_kws={'label': "Cohen's Kappa"},
    linewidths=0.5,
    linecolor='white'
)

# Adjust tick labels and layout.
ax.xaxis.tick_bottom()
plt.xticks(rotation=45)
plt.xlabel("Conditions and Global Agreement")
plt.ylabel("Conditions")
plt.title("Across-Condition Agreement with Global (Ground Truth) Agreement Scores")
plt.tight_layout()
plt.show()